# Main data manipulation and exploration

### Setup

In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import sys
from statsmodels import robust
import matplotlib.pyplot as plt
import os.path
import anndata as ad
import seaborn as sns
import matplotlib as mpl
import os
# import scvi


In [2]:
conda env list


Note: you may need to restart the kernel to use updated packages.


Traceback (most recent call last):
  File "C:\Users\sambe\anaconda3\Scripts\conda-script.py", line 11, in <module>
    from conda.cli import main
ModuleNotFoundError: No module named 'conda'


In [3]:
#os.chdir("..\\")
# should use forwardslash(/)as double backslash(\\) only works for windows, e.g.
bigdata = 'C:\\Users\\sambe\\Documents\\bigdata\\'
os.getcwd()

'C:\\Users\\sambe\\OneDrive - Newcastle University\\Research projects\\python\\scripts'

In [4]:
metadata = pd.read_csv(bigdata + 'IG_anno_lvl_2_final_clean_051121.csv', index_col=0)

panfoetal = sc.read(bigdata + "PAN.A01.v01.entire_data_raw_count.20210429.h5ad")

panfoetal.var_names_make_unique()
panfoetal

Only considering the two last: ['.20210429', '.h5ad'].
Only considering the two last: ['.20210429', '.h5ad'].


C:\Users\sambe\anaconda3\envs\research23\lib\site-packages\anndata\_core\anndata.py:1830: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


AnnData object with n_obs × n_vars = 911873 × 33538
    obs: 'n_counts', 'n_genes', 'file', 'mito', 'doublet_scores', 'predicted_doublets', 'name'
    var: 'GeneID', 'GeneName'

In [5]:
wholeart = sc.read(bigdata + 'PAN.A01.v01.raw_count.20210429.PFI.embedding.h5ad')

Only considering the two last: ['.embedding', '.h5ad'].
Only considering the two last: ['.embedding', '.h5ad'].


In [6]:
lymphart = sc.read(bigdata + 'PAN.A01.v01.raw_count.20210429.LYMPHOID.embedding.h5ad')

Only considering the two last: ['.embedding', '.h5ad'].
Only considering the two last: ['.embedding', '.h5ad'].


#### partial loading

In [7]:
#panfoetal = pd.read('C:\\Users\\sambe\\Documents\\bigdata\\PAN.A01.v01.entire_data_raw_count.20210429.h5ad', backed='r')
# backed arguement partially reads in the data due to its size, "adata.file.close()" to close this data.

In [8]:
#bigdata.isbacked

In [9]:
#panfoetal.file.close()

In [10]:
#panfoetal = ad.read('C:\\Users\\sambe\\Documents\\bigdata\\PAN.A01.v01.entire_data_raw_count.20210429.h5ad')
#panfoetal.var_names_make_unique() 

### stats

In [11]:
lymphart

AnnData object with n_obs × n_vars = 241950 × 33538
    obs: 'n_counts', 'n_genes', 'file', 'mito', 'doublet_scores', 'predicted_doublets', 'old_annotation_uniform', 'organ', 'Sort_id', 'age', 'method', 'donor', 'sex', 'Sample', 'scvi_clusters', 'is_maternal_contaminant', 'anno_lvl_2_final_clean', 'celltype_annotation'
    var: 'GeneID', 'GeneName', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'scvi_model_var'
    uns: 'dendrogram_scvi_clusters', 'scvi'
    obsm: 'X_scvi', 'X_umap'
    obsp: 'scvi_connectivities', 'scvi_distances'

In [12]:
#display(panfoetal)

#display(panfoetal.obs[:5])

#display(metadata)

#display(panfoetal.obs)

#display(lymphart.obs)

lymphart_names = lymphart.obs["anno_lvl_2_final_clean"].unique()
lymphart_names =list(lymphart_names)
display(lymphart_names)
# collection of all the unique marked cell types in the sample.
#display(panfoetal_l_copy.obs.anno_lvl_2_final_clean.unique())

#display(len(lymphart_names))
#display(len(metadata_names))

#panfoetal.X.max()

#help(panfoetal.X.max)

['LARGE_PRE_B',
 'LOW_Q_INCONSISTENT',
 'CYCLING_T',
 'PRE_PRO_B',
 'PRO_B',
 'CD4+T',
 'CD8+T',
 'CYCLING_B',
 'TREG',
 'LMPP_MLP',
 'NK',
 'CYCLING_NK',
 'ILC3',
 'ILC2',
 'SMALL_PRE_B',
 'MATURE_B',
 'MEMP',
 'B1',
 'CYCLING_ILC',
 'TYPE_1_INNATE_T',
 'IMMATURE_B',
 'LATE_PRO_B',
 'HIGH_MITO',
 'CYCLING_MPP',
 'DOUBLET_ERY_B',
 'HSC_MPP',
 'GMP',
 'EARLY_MK',
 'PROMYELOCYTE',
 'nan',
 'PROMONOCYTE',
 'DOUBLET',
 'CYCLING_MEMP',
 'DP(P)_T',
 'TYPE_3_INNATE_T',
 'CD8AA',
 'ABT(ENTRY)',
 'DP(Q)_T',
 'DN(P)_T',
 'PLASMA_B',
 'DN(early)_T',
 'CMP',
 'DN(Q)_T',
 'MEP']

### merging dataframes

##### Indexing portal data and raw - B cells and Progenitors

In [13]:
l1 = list(panfoetal.obs.index)
l2 = list(lymphart.obs.index)

In [14]:
print('the length of l1 - l2 is: ' + str(len(l1) - len(l2)) )

the length of l1 - l2 is: 669923


In [15]:
# are the lists the same len?
print('the length of l1 - l2 is: ' + str(len(l1) - len(l2)) )
print('the length of l1 is: ' + str(len(l1) ))
print('the length of l2 is: ' + str(len(l2)) )

the length of l1 - l2 is: 669923
the length of l1 is: 911873
the length of l2 is: 241950


In [16]:
print('the length of intersecting indexes between L1 and L2 is: ' + str(len(list(set(l1) & set(l2)))))

the length of intersecting indexes between L1 and L2 is: 241950


In [17]:
l1_missing = list(set(l2).difference(set(l1)))
l2_missing = list(set(l1).difference(set(l2)))

display(len(l1_missing))
display(len(l2_missing))

0

669923

In [18]:
'''l2_missing = list(panfoetal.obs[~panfoetal.obs.index.isin(inter)].index)
l1_missing = list(lymphart.obs[~lymphart.obs.index.isin(inter)].index)

display(len(l1_missing))
display(len(l2_missing))'''

'l2_missing = list(panfoetal.obs[~panfoetal.obs.index.isin(inter)].index)\nl1_missing = list(lymphart.obs[~lymphart.obs.index.isin(inter)].index)\n\ndisplay(len(l1_missing))\ndisplay(len(l2_missing))'

In [19]:
pan_immune_index = panfoetal[panfoetal.obs.index.isin(list(lymphart.obs.index))].copy()
pan_immune_index

AnnData object with n_obs × n_vars = 241950 × 33538
    obs: 'n_counts', 'n_genes', 'file', 'mito', 'doublet_scores', 'predicted_doublets', 'name'
    var: 'GeneID', 'GeneName'

In [20]:
pan_immune_index.obs

,n_counts,n_genes,file,mito,doublet_scores,predicted_doublets,name
FCAImmP7579224-CTAATGGCACTGTGTA,51305.0,5492,FCAImmP7579224,0.046467,0.176471,False,FCAImmP7579224_filtered.h5ad
FCAImmP7579224-ATTATCCAGAGAACAG,39999.0,5076,FCAImmP7579224,0.038651,0.087221,False,FCAImmP7579224_filtered.h5ad
FCAImmP7579224-GACGGCTAGCCACCTG,38114.0,5282,FCAImmP7579224,0.034633,0.110588,False,FCAImmP7579224_filtered.h5ad
FCAImmP7579224-GCGGGTTGTCCGAGTC,33207.0,4690,FCAImmP7579224,0.028247,0.133690,False,FCAImmP7579224_filtered.h5ad
FCAImmP7579224-AGTTGGTAGTGTTAGA,31058.0,4591,FCAImmP7579224,0.044433,0.133690,False,FCAImmP7579224_filtered.h5ad
...,...,...,...,...,...,...,...
FCAImmP7277565-GTCAAGTCATAAAGGT,2022.0,808,FCAImmP7277565,0.016320,0.026087,False,FCAImmP7277565_filtered.h5ad
FCAImmP7277565-ATCATGGGTAAATGTG,2021.0,847,FCAImmP7277565,0.021771,0.028971,False,FCAImmP7277565_filtered.h5ad
FCAImmP7277565-ATTGGTGCACGCTTTC,2013.0,946,FCAImmP7277565,0.022355,0.016225,False,FCAImmP7277565_filtered.h5ad
FCAImmP7277565-GAACATCGTACCAGTT,2018.0,913,FCAImmP7277565,0.019822,0.089376,False,FCAImmP7277565_filtered.h5ad


In [21]:
''' Above test of the .copy method to account for error found in the DEGs test in notebook "c1_marker_deduction.ipynb" '''

' Above test of the .copy method to account for error found in the DEGs test in notebook "c1_marker_deduction.ipynb" '

In [22]:
pan_immune_index.obs = lymphart.obs.copy()
pan_immune_index.var = lymphart.var.copy()
pan_immune_index

AnnData object with n_obs × n_vars = 241950 × 33538
    obs: 'n_counts', 'n_genes', 'file', 'mito', 'doublet_scores', 'predicted_doublets', 'old_annotation_uniform', 'organ', 'Sort_id', 'age', 'method', 'donor', 'sex', 'Sample', 'scvi_clusters', 'is_maternal_contaminant', 'anno_lvl_2_final_clean', 'celltype_annotation'
    var: 'GeneID', 'GeneName', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'scvi_model_var'

In [23]:
pan_immune_index.write('C:\\Users\\sambe\\Documents\\bigdata\\pan_immune_index.h5ad', compression="gzip")


In [24]:
bcells = ['PRE_PRO_B', 'PRE_PRO_B_CELL', 'PRE_PRO_B',  'LATE_PRO_B','PRO_B','LARGE_PRE_B','SMALL_PRE_B', 'IMMATURE_B','MATURE_B','B1', 'CYCLING_B']
pcells = ['HSC_MPP','LMPP_MLP','CYCLING_MPP']
stroma = [ 'DOUBLET_IMMUNE_FIBROBLAST', 'CYCLING_FIBROBLAST_I','FIBROBLAST_I','MYOFIBROBLAST','FIBROBLAST_XVII','FIBROBLAST_II','CYCLING_FIBROBLAST_II', 'FIBROBLAST_III', 
          'FIBROBLAST_IV','FIBROBLAST_V','FIBROBLAST_VI', 'FIBROBLAST_VII','FIBROBLAST_VIII', 'FIBROBLAST_IX','FIBROBLAST_X', 'FIBROBLAST_XI''FIBROBLAST_XII', 'FIBROBLAST_XIII',
          'FIBROBLAST_XIV','FIBROBLAST_XV','FIBROBLAST_XVI','CYCLING_FIBROBLAST_I', 'DOUBLETS_FIBRO_ERY', 'MYOFIBROBLAST_I', 'SMOOTH_MUSCLE', 'ENDOTHELIUM_I', 'ENDOTHELIUM_II',
           'ENDOTHELIUM_III','DOUBLET_ENDOTHELIUM_ERYTHROCYTE', 'ENDOTHELIUM_IV', 'ENDOTHELIUM_V','EPITHELIUM_I','CYCLING_EPITHELIUM','EPITHELIUM_II','SKELETAL_MUSCLE']

In [25]:
pan_immune_bcells = pan_immune_index[pan_immune_index.obs.anno_lvl_2_final_clean.isin(bcells)].copy()
pan_immune_pcells = pan_immune_index[pan_immune_index.obs.anno_lvl_2_final_clean.isin(pcells)].copy()
pan_immune_bcells.write(bigdata + 'pan_immune_bcells.h5ad', compression="gzip")
pan_immune_pcells.write(bigdata + 'pan_immune_pcells.h5ad', compression="gzip")

# pan_immune_bcells = pan_immune_index[pan_immune_index.obs.anno_lvl_2_final_clean.isin(stroma)]
# stroma isn't going to work as stroma data isn't in the lymphoid dataset



In [26]:
#pan_whole_index = panfoetal[panfoetal.obs.index.isin(list(wholeart.obs.index))]
#pan_whole_index

#pan_whole_index.obs = wholeart.obs
#pan_whole_index.var = wholeart.var
#pan_whole_index

### THIS SEGMENT IS AN ATTEMPT TO PRODUCE THE STORMAL DATASET. BUT DUE TO MEMORY SHORTAGE THE DATASET CAN'T BE CREATED ###

##### stroma attempt 2

In [27]:
#stromart = sc.read(bigdata + 'PAN.A01.v01.raw_count.20210429.STROMA.embedding.h5ad')


In [28]:
"""
stromart_names = stromart.obs.anno_lvl_2_final_clean.unique()
stromart_names =list(stromart_names)
display(len(stromart_names))
display(len(stroma))
display(stromart_names)
"""

'\nstromart_names = stromart.obs.anno_lvl_2_final_clean.unique()\nstromart_names =list(stromart_names)\ndisplay(len(stromart_names))\ndisplay(len(stroma))\ndisplay(stromart_names)\n'

In [29]:
"""
stromart = sc.read(bigdata + 'PAN.A01.v01.raw_count.20210429.STROMA.embedding.h5ad')
pan_stroma_index = panfoetal[panfoetal.obs.index.isin(list(stromart.obs.index))]
pan_stroma_index

pan_stroma_index.obs = stromart.obs
pan_stroma_index.var = stromart.var
pan_stroma_index
"""

"\nstromart = sc.read(bigdata + 'PAN.A01.v01.raw_count.20210429.STROMA.embedding.h5ad')\npan_stroma_index = panfoetal[panfoetal.obs.index.isin(list(stromart.obs.index))]\npan_stroma_index\n\npan_stroma_index.obs = stromart.obs\npan_stroma_index.var = stromart.var\npan_stroma_index\n"

In [30]:
"""
pan_stroma_cells = pan_stroma_index[pan_stroma_index.obs.anno_lvl_2_final_clean.isin(stroma)].copy()
pan_stroma_cells.write(bigdata + 'pan_stroma_cells.h5ad', compression="gzip")
"""

'\npan_stroma_cells = pan_stroma_index[pan_stroma_index.obs.anno_lvl_2_final_clean.isin(stroma)].copy()\npan_stroma_cells.write(bigdata + \'pan_stroma_cells.h5ad\', compression="gzip")\n'

#### indexing then joining the metadata and raw - Is this segment necessary?

In [31]:
panconmeta = panfoetal[panfoetal.obs.index.isin(list(metadata.index))].copy()
display(panconmeta)
panconmeta.obs = pd.concat([panconmeta.obs,metadata], axis=1,join="inner").copy()
#concatenate the two dataframes of maindata and additional metadata. Outer filter used over inner as otherwise error.

MemoryError: Unable to allocate 7.67 GiB for an array with shape (2059061010,) and data type float32

In [ ]:
missing = panconmeta.obs.isnull().sum()
print(missing)
# print the total amount of missing data from each column, it has been displayed that 3695 cells were already filtered before classification.

In [ ]:
panconmeta_names = panconmeta.obs["anno_lvl_2_final_clean"].unique()
panconmeta_names =list(panconmeta_names)
display(panconmeta_names)

In [ ]:
# these listed cells are recorded as from the full dataset obtained from authors to the metadata also provided by authors. There is slightly less cells in the full set as opposed to the metadata and that should be remembered. 
# But both are much larger in size than the dataset from the portal e.g. 5gb portal full vs 15gb authors full.

bcells = ['PRE_PRO_B', 'PRE_PRO_B_CELL', 'PRE_PRO_B',  'LATE_PRO_B','PRO_B','LARGE_PRE_B','SMALL_PRE_B', 'IMMATURE_B','MATURE_B','B1', 'CYCLING_B']
pcells = ['HSC_MPP','LMPP_MLP','CYCLING_MPP']
stroma = [ 'DOUBLET_IMMUNE_FIBROBLAST', 'CYCLING_FIBROBLAST_I','FIBROBLAST_I','MYOFIBROBLAST','FIBROBLAST_XVII','FIBROBLAST_II','CYCLING_FIBROBLAST_II', 'FIBROBLAST_III', 
          'FIBROBLAST_IV','FIBROBLAST_V','FIBROBLAST_VI', 'FIBROBLAST_VII','FIBROBLAST_VIII', 'FIBROBLAST_IX','FIBROBLAST_X', 'FIBROBLAST_XI''FIBROBLAST_XII', 'FIBROBLAST_XIII',
          'FIBROBLAST_XIV','FIBROBLAST_XV','FIBROBLAST_XVI','CYCLING_FIBROBLAST_I', 'DOUBLETS_FIBRO_ERY', 'MYOFIBROBLAST_I', 'SMOOTH_MUSCLE', 'ENDOTHELIUM_I', 'ENDOTHELIUM_II',
           'ENDOTHELIUM_III','DOUBLET_ENDOTHELIUM_ERYTHROCYTE', 'ENDOTHELIUM_IV', 'ENDOTHELIUM_V','EPITHELIUM_I','CYCLING_EPITHELIUM','EPITHELIUM_II','SKELETAL_MUSCLE']


#stroma = ['not_ready_yet_tey_ydaer_ton']

#panfoetal_copy = panfoetal[panfoetal.obs.anno_lvl_2_final_clean.isin(bcells)]
#print(panfoetal_copy)
panconmeta_l = panconmeta[panconmeta.obs.anno_lvl_2_final_clean.isin(lcells)].copy()
print(panconmeta_l)
lymphart = lymphart[lymphart.obs.anno_lvl_2_final_clean.isin(lcells)].copy()
print(lymphart)

######### THE NUMBER OF CELLS IN THE MAIN IS LOWER THAN THE PORTAL, SOMETHING NEEDS LOOKING INTO, MAYBE INDEX THE MAIN TO THE PORTAL METADATA

### Observations / comparisons

In [ ]:
"""
display(panconmeta_l.obs)

display(panconmeta_l.obs)

display(241950 - 249463)

display(panconmeta_l.obs.anno_lvl_2_final_clean.nunique())

display(lymphart.obs.anno_lvl_2_final_clean.nunique())
"""

In [ ]:
"""
alist = panconmeta_l.obs.anno_lvl_2_final_clean.value_counts()
blist = lymphart.obs.anno_lvl_2_final_clean.value_counts()
#print(a == b)
#print(sorted(a) == sorted(b))
d = {'panconmeta_l':alist,'lymph':blist}
d = pd.DataFrame(d)
d['panconmeta_l'] = d['panconmeta_l'].fillna(0)
d = d.astype({'panconmeta_l':'int'})
#d["ss"] = "panfoetal_l" == "lymph" #attempt to filter the non-matching values
values = d.panconmeta_l - d.lymph
d['values'] = values
display(d.loc[values == 0])
display(d.loc[values != 0])

# comparison of the provided lymphoid dataset and panfoetal cut-
# -to display the difference Identical values have been removed.
"""

In [ ]:
panfoetal.obs.anno_lvl_2_final_clean.unique()

###### with antony help

In [ ]:
#panfoetal.obs['anno'] = metadata['anno_lvl_2_final_clean'].copy()

In [ ]:
#bcells = panfoetal[panfoetal.obs['anno'].isin(lcells)].copy()

In [ ]:
#lymphart.obs

In [ ]:
#celltypes = ['PRE_PRO_B_CELL','PRO_B','B1']
#Bcells = Panfetal[Panfetal.obs['anno'].isin(celltypes)]

### aww

In [ ]:
#sc.pl.highest_expr_genes(panfoetal_l_copy, n_top=20, )


### Quality control?

###### Unsure how to proceed due to past researchers efforts.

### GRAPHH

In [ ]:
#panfoetal_l_copy.var['mt'] = panfoetal_l_copy.var_names.str.startswith('MT-').copy()  # annotate the group of mitochondrial genes as 'mt'
#sc.pp.calculate_qc_metrics(panfoetal_l_copy, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)


In [ ]:
#panfoetal_l_copy.var_names
#sc.pp.filter_cells(panfoetal_l_copy, min_genes=200)
#sc.pp.filter_genes(panfoetal_l_copy, min_cells=3)


In [ ]:
#sc.pl.violin(panfoetal_l_copy, ['total_counts'],
#             jitter=0.4, multi_panel=True)